In [30]:
%pip install -q kagglehub[pandas-datasets] scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import numpy as np
import pandas as pd
import tensorflow as tf
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [32]:
# Load dataset from KaggleHub
file_path = "all-data.csv" 
df = None
for enc in ["latin-1", "cp1252", "utf-8"]:
    try:
        df = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "ankurzing/sentiment-analysis-for-financial-news",
            file_path,
            pandas_kwargs={
                "encoding": enc,
                "header": None,
                "names": ["sentiment", "sentence"],
            },
        )
        print(f"Loaded dataset using encoding: {enc}")
        break
    except Exception as e:
        print(f"Failed with encoding={enc}: {e}")

if df is None:
    raise ValueError("Unable to load dataset. Check file_path and Kaggle credentials.")

print("First 5 records:")
print(df.head())
print("\nColumns:", list(df.columns))
print("\nShape:", df.shape)

C:\Users\Swornim\AppData\Local\Temp\ipykernel_23848\1385383747.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Loaded dataset using encoding: latin-1
First 5 records:
  sentiment                                           sentence
0   neutral  According to Gran , the company has no plans t...
1   neutral  Technopolis plans to develop in stages an area...
2  negative  The international electronic industry company ...
3  positive  With the new production plant the company woul...
4  positive  According to the company 's updated strategy f...

Columns: ['sentiment', 'sentence']

Shape: (4846, 2)


In [ ]:
binary_df = df.copy()

binary_df["sentence"] = binary_df["sentence"].astype(str)
binary_df["sentiment"] = binary_df["sentiment"].astype(str).str.strip().str.lower()
binary_df = binary_df[binary_df["sentiment"].isin(["positive", "negative"])].copy()
binary_df["target"] = (binary_df["sentiment"] == "positive").astype(int)

X_text = binary_df["sentence"].values
y = binary_df["target"].values

# Train/Test split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=SEED, stratify=y
)

print("Label distribution (binary):")
print(binary_df["sentiment"].value_counts())
print("Training samples:", len(X_train_text))
print("Test samples:", len(X_test_text))

Label distribution (binary):
sentiment
positive    1363
negative     604
Name: count, dtype: int64
Training samples: 1573
Test samples: 394


In [34]:
# Text to padded integer sequences
VOCAB_SIZE = 8000
MAX_LENGTH = 40

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LENGTH, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LENGTH, padding="post", truncating="post")

print("X_train shape:", X_train_pad.shape)
print("X_test shape:", X_test_pad.shape)

X_train shape: (1573, 40)
X_test shape: (394, 40)


In [35]:
# Build BiLSTM model
model = Sequential([
    tf.keras.Input(shape=(MAX_LENGTH,)),
    Embedding(input_dim=VOCAB_SIZE, output_dim=128),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 40, 128)        │     1,024,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,126,977 (4.30 MB)

 Trainable params: 1,126,977 (4.30 MB)

 Non-trainable params: 0 (0.00 B)

In [36]:
# Train
history = model.fit(
    X_train_pad,
    y_train,
    epochs=8,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/8
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - accuracy: 0.6709 - loss: 0.6284 - val_accuracy: 0.7429 - val_loss: 0.5225
Epoch 2/8
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.8116 - loss: 0.3983 - val_accuracy: 0.8190 - val_loss: 0.5542
Epoch 3/8
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9722 - loss: 0.0970 - val_accuracy: 0.8444 - val_loss: 0.5794
Epoch 4/8
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9801 - loss: 0.0572 - val_accuracy: 0.8508 - val_loss: 0.5220
Epoch 5/8
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.9897 - loss: 0.0300 - val_accuracy: 0.8413 - val_loss: 0.5168
Epoch 6/8
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9928 - loss: 0.0179 - val_accuracy: 0.8413 - val_loss: 0.5818
Epoch 7/8
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9952 - loss: 0.0116 - val_accuracy: 0.8413 - val_loss: 0.6568
Epoch 8/8
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.9992 - loss: 0.0039 - val_accuracy: 0.8413 - val_loss:

In [37]:
# Evaluate
test_loss, test_acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

y_prob = model.predict(X_test_pad, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 81.98%

Classification Report:
              precision    recall  f1-score   support

    negative       0.70      0.73      0.71       121
    positive       0.88      0.86      0.87       273

    accuracy                           0.82       394
   macro avg       0.79      0.79      0.79       394
weighted avg       0.82      0.82      0.82       394

Confusion Matrix:
[[ 88  33]
 [ 38 235]]


In [38]:
# Inference on custom financial headlines
samples = [
    "Company posts record quarterly profit and raises guidance",
    "Shares plunge after weak earnings and lower forecast",
    "Bank reports stable results despite market volatility"
]

sample_seq = tokenizer.texts_to_sequences(samples)
sample_pad = pad_sequences(sample_seq, maxlen=MAX_LENGTH, padding="post", truncating="post")
sample_prob = model.predict(sample_pad, verbose=0).ravel()

for text, p in zip(samples, sample_prob):
    label = "positive" if p >= 0.5 else "negative"
    print(f"Headline: {text}")
    print(f"Predicted sentiment: {label} (score={p:.3f})\n")

Headline: Company posts record quarterly profit and raises guidance
Predicted sentiment: positive (score=0.993)

Headline: Shares plunge after weak earnings and lower forecast
Predicted sentiment: negative (score=0.001)

Headline: Bank reports stable results despite market volatility
Predicted sentiment: negative (score=0.421)

